# Single signal analysis

This notebook performs a statistical analysis of individual seismic acceleration signals.
For each signal, the analysis covers: empirical probability density functions, 
Gaussian fit and normality assessment, heavy-tail characterization, 
moment scaling analysis, and displacement autocorrelation functions.

## 1. Imports and visualization settings

In [ ]:
from pathlib import Path
import pandas as pd
import os
from src.signals import (compute_moment_scaling_acc,compute_moment_scaling_vel, compute_moment_scaling_disp,
                         compute_scaling_exponents,test_scaling_linearity, fit_piecewise_scaling,
                         compute_displacement_autocorrelation, analyze_autocorrelation_scaling)
from src.plot_settings import set_plot_style
colors = set_plot_style()

## 2. Data loading

Preprocessed acceleration data are loaded from the parquet file produced 
in notebook 02. Each signal has been baseline-corrected and normalized 
by its standard deviation, so that $\bar{a} = 0$ and $\sigma_a = 1$.

In [2]:
# Load preprocessed data
df_acc_clean = pd.read_parquet('../data/processed/acc_preprocessed_all.parquet')
print("Shape:", df_acc_clean.shape)
display(df_acc_clean.head())

Shape: (2614815, 4)


,file,sample,acceleration,acceleration_normalized
0,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,0,-6.666667e-10,-3.401661e-08
1,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,1,-6.666667e-10,-3.401661e-08
2,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,2,-6.666667e-10,-3.401661e-08
3,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,3,-6.666667e-10,-3.401661e-08
4,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,4,-6.666667e-10,-3.401661e-08


In [3]:
# Figures output directory
FIGURES_DIR = Path('../figures/03_single_signal')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 4. Moment scaling analysis - acceleration

### 4.1 Computation of q-th order moments

Moments are computed for a range of moment orders q and time scales tau. 
The resulting dataframe contains one row per $(\text{file}, q, \tau)$ combination.

In [4]:
q_values_acc = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
tau_values_acc = [10, 50, 100, 200, 500, 1000, 2000, 5000, 10000]

# Stations excluded from moment scaling due to short signal length
short_stations = ['SURF', 'BRZ', 'BHB', 'CRI', 'SLZ', 'SAV']

df_acc_scaling = df_acc_clean[
    ~df_acc_clean['file'].str.split('.').str[1].isin(short_stations)
].copy()

print(f"Signals used for moment scaling: {df_acc_scaling['file'].nunique()} / {df_acc_clean['file'].nunique()}")

Signals used for moment scaling: 48 / 66


In [5]:
df_moments_acc = compute_moment_scaling_acc(df_acc_scaling, q_values_acc, tau_values_acc)
print(df_moments_acc.shape)
display(df_moments_acc)

(3456, 6)


,file,station,stream,q,tau,moment
0,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,0.5,10,0.591691
1,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,1.0,10,0.587259
2,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,1.5,10,0.938694
3,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,2.0,10,2.129010
4,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,2.5,10,5.949601
...,...,...,...,...,...,...
3451,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,2.0,10000,2.524536
3452,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,2.5,10000,5.608439
3453,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,3.0,10000,13.615232
3454,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,3.5,10000,35.341492


### 4.2 Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

The slope $\zeta(q)$ is the scaling exponent. A plot of $\zeta(q)$ vs $q$ 
is produced for each signal, with the reference line $\zeta(q) = q/2$ 
corresponding to normal diffusion.

In [6]:
df_exponents_acc = compute_scaling_exponents(df_moments_acc, output_dir=FIGURES_DIR / 'scaling'/ 'acceleration' / 'exponents')

Saved: 48/48 individual plots
All individual plots saved successfully!


In [7]:
# Save results
try:
    df_exponents_acc.to_parquet('../data/processed/scaling_exponents_acc.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


### 4.3 Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing two models 
using AIC:

- **Linear**: $\zeta(q) = aq + b$
- **Quadratic**: $\zeta(q) = aq^2 + bq + c$

If the quadratic model is preferred, the scaling is non-linear, 
indicating anomalous diffusion.

A piecewise linear fit is also performed to detect the presence of 
two distinct scaling regimes, consistent with the framework of strong 
anomalous diffusion (Vollmer et al., 2021):

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes. The breakpoint $q^*$ is estimated by minimizing the 
total residual sum of squares.

In [8]:
# Linearity check
df_linearity_acc = test_scaling_linearity(df_exponents_acc, output_dir=FIGURES_DIR / 'scaling'/ 'acceleration' / 'linearity')

Saved: 48/48 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit
quadratic    46
linear        2
Name: count, dtype: int64


In [9]:
# Save results
try:
    df_linearity_acc.to_parquet('../data/processed/scaling_linearity_acc.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


In [10]:
# Piecewise linear fit
df_piecewise_acc = fit_piecewise_scaling(df_exponents_acc, output_dir=FIGURES_DIR / 'scaling'/ 'acceleration' / 'piecewise')

Saved: 48/48 individual plots
All individual plots saved successfully!


In [11]:
# Save results
try:
    df_piecewise_acc.to_parquet('../data/processed/scaling_piecewise_acc.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


## 5. Moment scaling analysis - velocity

### 5.1 Computation of q-th order moments

Moments are computed for a range of moment orders q and time scales tau. 
The resulting dataframe contains one row per $(\text{file}, q, \tau)$ combination.

In [12]:
q_values_vel = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
tau_values_vel = [10, 50, 100, 200, 500, 1000, 2000, 5000, 10000]

# Stations excluded from moment scaling due to short signal length
short_stations = ['SURF', 'BRZ', 'BHB', 'CRI', 'SLZ', 'SAV']

df_vel_scaling = df_acc_clean[
    ~df_acc_clean['file'].str.split('.').str[1].isin(short_stations)
].copy()

print(f"Signals used for moment scaling: {df_vel_scaling['file'].nunique()} / {df_acc_clean['file'].nunique()}")

Signals used for moment scaling: 48 / 66


In [13]:
df_moments_vel = compute_moment_scaling_vel(df_vel_scaling, q_values_vel, tau_values_vel)
print(df_moments_vel.shape)
display(df_moments_vel)

(3456, 6)


,file,station,stream,q,tau,moment
0,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,0.5,10,0.075614
1,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,1.0,10,0.011713
2,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,1.5,10,0.003076
3,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,2.0,10,0.001103
4,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,2.5,10,0.000480
...,...,...,...,...,...,...
3451,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,2.0,10000,0.006666
3452,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,2.5,10000,0.003654
3453,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,3.0,10000,0.002297
3454,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,3.5,10000,0.001593


### 5.2 Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

The slope $\zeta(q)$ is the scaling exponent. A plot of $\zeta(q)$ vs $q$ 
is produced for each signal, with the reference line $\zeta(q) = q/2$ 
corresponding to normal diffusion.

In [14]:
df_exponents_vel = compute_scaling_exponents(df_moments_vel, output_dir=FIGURES_DIR / 'scaling'/ 'velocity' / 'exponents')

Saved: 48/48 individual plots
All individual plots saved successfully!


In [15]:
# Save results
try:
    df_exponents_vel.to_parquet('../data/processed/scaling_exponents_vel.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


### 5.3 Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing two models 
using AIC:

- **Linear**: $\zeta(q) = aq + b$
- **Quadratic**: $\zeta(q) = aq^2 + bq + c$

If the quadratic model is preferred, the scaling is non-linear, 
indicating anomalous diffusion.

A piecewise linear fit is also performed to detect the presence of 
two distinct scaling regimes, consistent with the framework of strong 
anomalous diffusion (Vollmer et al., 2021):

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes. The breakpoint $q^*$ is estimated by minimizing the 
total residual sum of squares.

In [16]:
# Linearity check
df_linearity_vel = test_scaling_linearity(df_exponents_vel, output_dir=FIGURES_DIR / 'scaling'/ 'velocity' / 'linearity')

Saved: 48/48 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit
quadratic    47
linear        1
Name: count, dtype: int64


In [17]:
# Save results
try:
    df_linearity_vel.to_parquet('../data/processed/scaling_linearity_vel.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


In [18]:
# Piecewise linear fit
df_piecewise_vel = fit_piecewise_scaling(df_exponents_vel, output_dir=FIGURES_DIR / 'scaling'/ 'velocity' / 'piecewise')

Saved: 48/48 individual plots
All individual plots saved successfully!


In [19]:
# Save results
try:
    df_piecewise_vel.to_parquet('../data/processed/scaling_piecewise_vel.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


## 6. Moment scaling analysis - 

### 6.1 Computation of q-th order moments

Moments are computed for a range of moment orders q and time scales tau. 
The resulting dataframe contains one row per $(\text{file}, q, \tau)$ combination.

In [20]:
q_values_disp = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0]
tau_values_disp = [10, 50, 100, 200, 500, 1000, 2000, 5000, 10000]

# Stations excluded from moment scaling due to short signal length
short_stations = ['SURF', 'BRZ', 'BHB', 'CRI', 'SLZ', 'SAV']

df_disp_scaling = df_acc_clean[
    ~df_acc_clean['file'].str.split('.').str[1].isin(short_stations)
].copy()

print(f"Signals used for moment scaling: {df_disp_scaling['file'].nunique()} / {df_acc_clean['file'].nunique()}")

Signals used for moment scaling: 48 / 66


In [21]:
df_moments_disp = compute_moment_scaling_disp(df_disp_scaling, q_values_disp, tau_values_disp)
print(df_moments_disp.shape)
display(df_moments_disp)

(3456, 6)


,file,station,stream,q,tau,moment
0,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,0.5,10,1.647991e-02
1,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,1.0,10,5.616708e-04
2,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,1.5,10,3.113940e-05
3,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,2.0,10,2.305301e-06
4,FR.EILF.00.HNE.D.INT-41004391.ACC.MP.ASC,EILF,HNE,2.5,10,2.050691e-07
...,...,...,...,...,...,...
3451,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,2.0,10000,5.404816e-05
3452,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,2.5,10000,8.313803e-06
3453,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,3.0,10000,1.481448e-06
3454,RA.MENA.00.HNZ.D.INT-41004391.ACC.MP.ASC,MENA,HNZ,3.5,10000,2.942947e-07


### 6.2 Scaling exponents estimation

For each signal and each moment order $q$, the scaling exponent $\zeta(q)$ 
is estimated by linear regression of $\log M_q(\tau)$ vs $\log \tau$:

$$\log M_q(\tau) = \zeta(q) \log \tau + c$$

The slope $\zeta(q)$ is the scaling exponent. A plot of $\zeta(q)$ vs $q$ 
is produced for each signal, with the reference line $\zeta(q) = q/2$ 
corresponding to normal diffusion.

In [22]:
df_exponents_disp = compute_scaling_exponents(df_moments_disp, output_dir=FIGURES_DIR / 'scaling'/ 'displacement' / 'exponents')

Saved: 48/48 individual plots
All individual plots saved successfully!


In [23]:
# Save results
try:
    df_exponents_disp.to_parquet('../data/processed/scaling_exponents_disp.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


### 6.3 Linearity check

The linearity of $\zeta(q)$ vs $q$ is assessed by comparing two models 
using AIC:

- **Linear**: $\zeta(q) = aq + b$
- **Quadratic**: $\zeta(q) = aq^2 + bq + c$

If the quadratic model is preferred, the scaling is non-linear, 
indicating anomalous diffusion.

A piecewise linear fit is also performed to detect the presence of 
two distinct scaling regimes, consistent with the framework of strong 
anomalous diffusion (Vollmer et al., 2021):

$$\zeta(q) = \begin{cases} \phi q & q \leq q^* \\ \lambda q - q^*(\lambda - \phi) & q > q^* \end{cases}$$

where $q^*$ is the breakpoint and $\phi$, $\lambda$ are the slopes of 
the two regimes. The breakpoint $q^*$ is estimated by minimizing the 
total residual sum of squares.

In [24]:
# Linearity check
df_linearity_disp = test_scaling_linearity(df_exponents_disp, output_dir=FIGURES_DIR / 'scaling'/ 'displacement' / 'linearity')

Saved: 48/48 individual plots
All individual plots saved successfully!

Best fit by AIC:
best_fit
quadratic    48
Name: count, dtype: int64


In [25]:
# Save results
try:
    df_linearity_disp.to_parquet('../data/processed/scaling_linearity_disp.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


In [26]:
# Piecewise linear fit
df_piecewise_disp = fit_piecewise_scaling(df_exponents_disp, output_dir=FIGURES_DIR / 'scaling'/ 'displacement' / 'piecewise')

Saved: 48/48 individual plots
All individual plots saved successfully!


In [27]:
# Save results
try:
    df_piecewise_disp.to_parquet('../data/processed/scaling_piecewise_disp.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

Saved successfully!


## 5. Displacement autocorrelation functions

The displacement autocorrelation function is computed for each signal 
following the framework of Vollmer et al. (2021):

$$C(t_1, t_2) = \langle (a(t_1) - a_0)(a(t_2) - a_0) \rangle$$

where $a_0 = a(0)$ is the initial value of the signal. The function 
is evaluated on a logarithmic grid of $(t_1, t_2)$ pairs to capture 
scaling behavior across multiple time scales.

### 5.1 Computation

Autocorrelation functions are computed on a logarithmic grid of $n_{points}$ values for both $t_1$ and $t_2$, resulting in a matrix of size $n_{points} \times n_{points}$ for each signal.

In [ ]:
df_autocorr, C_matrices = compute_displacement_autocorrelation(df_acc_clean, output_dir=FIGURES_DIR / 'autocorrelation')

### 5.2 Scaling behavior

The scaling exponent $\beta$ is estimated from the diagonal $C(t, t)$, 
which is expected to scale as:

$$C(t, t) \sim t^\beta$$

The exponent $\beta$ is estimated by linear regression of 
$\log |C(t, t)|$ vs $\log t$. A value of $\beta = 1$ is consistent 
with normal diffusion, while $\beta \neq 1$ indicates anomalous scaling.

In [ ]:
df_autocorr_scaling = analyze_autocorrelation_scaling(
    C_matrices, output_dir=FIGURES_DIR / 'autocorrelation')

In [ ]:
# Save results
try:
    df_autocorr_scaling.to_parquet('../data/processed/autocorr_scaling.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")

## 6. Summary

A summary table collects the main results from all analyses for each signal:

| Column | Description |
|--------|-------------|
| `kurtosis` | Excess kurtosis $\kappa$ |
| `skewness` | Skewness $\gamma$ |
| `non_gaussian` | Anderson-Darling test result ($\alpha = 0.05$) |
| `best_fit_aic` | Best fitting distribution by AIC |
| `student_t_df` | Student-t degrees of freedom $\nu$ |
| `power_law_exp` | Power law exponent $\hat{\alpha}$ (Hill estimator) |
| `q_break` | Piecewise scaling breakpoint $q^*$ |
| `slope_low` | Scaling slope for $q \leq q^*$ |
| `slope_high` | Scaling slope for $q > q^*$ |
| `beta` | Autocorrelation scaling exponent $\beta$ |

In [ ]:
df_gaussian_results  = pd.read_parquet('../data/processed/gaussian_fit_results.parquet')
df_heavy_tail_results = pd.read_parquet('../data/processed/heavy_tail_results.parquet')
df_piecewise         = pd.read_parquet('../data/processed/scaling_piecewise.parquet')
df_autocorr_scaling  = pd.read_parquet('../data/processed/autocorr_scaling.parquet')

df_summary = df_gaussian_results[['station', 'stream', 'kurtosis', 'skewness', 'non_gaussian']]\
    .merge(df_heavy_tail_results[['station', 'stream', 'best_fit_aic', 'student_t_df', 'power_law_exp']],
           on=['station', 'stream'])\
    .merge(df_piecewise[['station', 'stream', 'q_break', 'slope_low', 'slope_high']],
           on=['station', 'stream'])\
    .merge(df_autocorr_scaling[['station', 'stream', 'beta', 'r2']],
           on=['station', 'stream'])

display(df_summary)

try:
    df_summary.to_parquet('../data/processed/summary.parquet', index=False)
    print("Saved successfully!")
except Exception as e:
    print(f"Error saving file: {e}")